# 02 · 인접 block 병합 평가

같은 문장이 서로 다른 block granularity로 출력될 때 인접 병합이 벌점을 어떻게 줄이는지 살핀다. 공식 OmniDocBench matcher의 축약형이다.

**학습 목표**: over-segmentation된 인접 block을 병합해 1:1 matching 편향을 완화하는 과정을 구현한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 외부 패키지는 없으며 Python 표준 기능만 사용한다.

In [ ]:
# 한 행만 유지하는 동적 계획법으로 전체 거리 행렬의 메모리를 쓰지 않는다.
def levenshtein(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        new = [i]
        for j, cb in enumerate(b, 1):
            new.append(min(new[-1] + 1, row[j] + 1, row[j - 1] + (ca != cb)))
        row = new
    return row[-1]

def distance(a, b):
    return levenshtein(a.casefold(), b.casefold()) / max(1, len(a), len(b))

def best_adjacent_match(reference, predicted_blocks, max_merge=3):
    candidates = []
    for start in range(len(predicted_blocks)):
        for width in range(1, min(max_merge, len(predicted_blocks) - start) + 1):
            merged = ' '.join(predicted_blocks[start:start + width])
            candidates.append((distance(reference, merged), start, width, merged))
    return min(candidates)


In [ ]:
reference = 'Revenue increased by 12 percent.'
predicted = ['Revenue increased', 'by 12 percent.', 'Next paragraph']
direct = min(distance(reference, block) for block in predicted)
merged = best_adjacent_match(reference, predicted)
print('best 1:1 distance:', round(direct, 3))
print('best adjacent merge:', merged)
assert merged[0] < direct
assert merged[2] == 2


인접 병합은 over-segmentation을 완화하지만 멀리 떨어진 조각, many-to-many 대응과 table-to-text 변환에는 부족하다. 이것이 후속 MGAM 같은 다중 입도 matcher가 필요한 이유다.